# Text2Cypher Toolkit: Kaggle Pipeline (parallel lanes)

Runs up to **two independent experiments in parallel**, one per T4 (a *lane*). Each lane is a separate process with its own GPU, config and Hub run: fine-tuning, inference, evaluation and GGUF export. Lanes share only the repo checkout, the dependency install and the base-model download.

Each lane keeps the single-GPU maths of the Colab runs (batch 1 x accumulation 16), so results stay comparable and Hub checkpoints resume across platforms.

### One-time setup per Kaggle account
1. Verify the account by phone (required for GPU and internet).
2. **Settings > Accelerator > GPU T4 x2.** Not P100.
3. **Settings > Internet > On.**
4. **Add-ons > Secrets:** add `HF_TOKEN` (write access) and attach it to this notebook.

### Running
- Set `INSTANCE_LABEL` and `LANES` in the configuration cell (the three standard instances are listed there).
- Use **Save Version > Save & Run All (Commit)** for unattended runs.
- Lane output is not streamed to the cell (two interleaved streams are unreadable). A heartbeat prints every `HEARTBEAT_S` seconds with each lane's step, latest log line, GPU memory and free RAM.
- Lane starts are staggered by `LANE_STAGGER_S` so model loading and tokenization do not compete for the 4 vCPUs at the same moment.
- Merge and GGUF export run one lane at a time (disk space, shared llama.cpp build). Merged fp16 weights are deleted after conversion unless `KEEP_MERGED` is set.

### Time budget
- Kaggle GPU sessions are capped at about 12 h. Each lane's training limit is the smaller of `TIME_LIMIT_MINUTES` and the session time left after `RESERVE_MINUTES`.
- A lane that pauses at its limit has its checkpoint on the Hub; it skips its remaining steps while the other lane continues. Commit a new version to resume. Lanes whose final adapter is on the Hub skip training.

### Logs, results and Hub commits
- Logs are **not** uploaded to the Hub (commit rate limits with several instances running). They are written to `OUTPUT_DIR/text2cypher-logs/<instance>/<session>/<lane>/` and saved with the notebook version.
- Each lane uploads its results folder to `results/<lane>/<session>/` in **one** commit when it ends (`UPLOAD_RESULTS_TO_HUB`). GGUF files are one commit each. Checkpoint sync is unchanged; training also mirrors `metrics.jsonl` to `runs/<run>/telemetry/<session>/` at most once an hour.

### Notifications
- With `NTFY_TOPIC` set, alerts go to `<NTFY_SERVER>/<NTFY_TOPIC>`: pipeline start, per-lane training start, step failures, time-limit pauses, per-lane completion with metrics, pipeline end, and kernel shutdown before the end.
- A hard kill at the session cap may not deliver an alert. ntfy.sh topics are public; only status text and log tails are sent.

In [ ]:
# 1. Pipeline configuration (edit these values, then Save Version > Save & Run All)
import time

PIPELINE_T0 = time.time()  # Session time budget is measured from here

# One lane per GPU: (model, schema variant, GPU index). Standard instances:
#   kaggle-gemma.ipynb   : INSTANCE_LABEL = "gemma-pair"
#                          LANES = [("gemma-4-e2b", "enhanced", "0"), ("gemma-4-e2b", "pruned", "1")]
#   kaggle-qwen.ipynb    : INSTANCE_LABEL = "qwen-pair"
#                          LANES = [("qwen3.5-2b", "enhanced", "0"), ("qwen3.5-2b", "pruned", "1")]
#   kaggle-granite.ipynb : INSTANCE_LABEL = "granite"
#                          LANES = [("granite-4.2-3b", "enhanced", "0")]
INSTANCE_LABEL = "gemma-pair"
LANES = [
    ("gemma-4-e2b", "enhanced", "0"),
    ("gemma-4-e2b", "pruned", "1"),
]

HUB_REPO = "2l3c34o1/text2cypher-checkpoints"
GITHUB_REPO = "https://github.com/Cxmrykk/capstone.git"

SESSION_BUDGET_MINUTES = 720     # Kaggle GPU session cap (12 h)
RESERVE_MINUTES = 90             # Setup before the training clock + upload + predict/eval/export
TIME_LIMIT_MINUTES = 600         # Upper bound on training wall-clock time per session
SKIP_TRAINING_IF_FINAL_EXISTS = True
LANE_STAGGER_S = 90              # Delay between lane starts (CPU-heavy loading/tokenization)

PREDICT_LIMIT = 100
RUN_EXECUTION_EVAL = True
RUN_EXPORT = True
UPLOAD_GGUF = True
KEEP_MERGED = False              # Delete merged fp16 weights after GGUF conversion (disk)
UPLOAD_RESULTS_TO_HUB = True     # One commit per lane at the end; logs are never uploaded

HEARTBEAT_S = 300
NTFY_TOPIC = "cxmrykk"           # Blank disables notifications
NTFY_SERVER = "https://ntfy.sh"

SCRATCH_DIR = "/kaggle/temp"     # Repo, models, artifacts (not saved; falls back to /tmp)
OUTPUT_DIR = "/kaggle/working"   # Logs and small results (saved with the notebook version)

CONFIGS = {
    ("gemma-4-e2b", "enhanced"): "configs/gemma-4-e2b.yaml",
    ("gemma-4-e2b", "pruned"): "configs/gemma-4-e2b-pruned.yaml",
    ("qwen3.5-2b", "enhanced"): "configs/qwen3.5-2b.yaml",
    ("qwen3.5-2b", "pruned"): "configs/qwen3.5-2b-pruned.yaml",
    ("granite-4.2-3b", "enhanced"): "configs/granite-4.2-3b.yaml",
}
DOWNLOAD_TARGETS = {"gemma-4-e2b": "gemma", "qwen3.5-2b": "qwen", "granite-4.2-3b": "granite"}

LANE_SPECS = []
for _model, _variant, _gpu in LANES:
    if (_model, _variant) not in CONFIGS:
        raise ValueError(f"No config for {_model} / {_variant} (Granite is enhanced-only).")
    LANE_SPECS.append({
        "key": f"{_model}-{_variant}",
        "model": _model,
        "variant": _variant,
        "gpu": str(_gpu),
        "config": CONFIGS[(_model, _variant)],
        "download_target": DOWNLOAD_TARGETS[_model],
    })
if not LANE_SPECS:
    raise ValueError("LANES is empty.")
if len(LANE_SPECS) > 2:
    raise ValueError("Kaggle T4 x2 has two GPUs; use at most two lanes.")
if len({lane["key"] for lane in LANE_SPECS}) != len(LANE_SPECS):
    raise ValueError("Each lane must be a different model/variant.")
if len({lane["gpu"] for lane in LANE_SPECS}) != len(LANE_SPECS):
    raise ValueError("Each lane needs its own GPU index.")

# Sequential downloads (history/006.md); a shared base model is downloaded once.
DOWNLOAD_ORDER = list(dict.fromkeys(lane["download_target"] for lane in LANE_SPECS))
# Only the Qwen configs use Unsloth.
INSTALL_UNSLOTH = any(lane["model"] == "qwen3.5-2b" for lane in LANE_SPECS)

print(f"Instance         : {INSTANCE_LABEL}")
for lane in LANE_SPECS:
    print(f"  Lane GPU {lane['gpu']}     : {lane['key']} ({lane['config']})")
print(f"Downloads        : {', '.join(DOWNLOAD_ORDER)}")
print(f"Install Unsloth  : {INSTALL_UNSLOTH}")
print(f"Hub repo         : {HUB_REPO} (checkpoints; results={UPLOAD_RESULTS_TO_HUB}; logs never)")
print(f"Session budget   : {SESSION_BUDGET_MINUTES} min (reserve {RESERVE_MINUTES} min, "
      f"training cap {TIME_LIMIT_MINUTES} min)")
print(f"Notifications    : {NTFY_SERVER.rstrip('/')}/{NTFY_TOPIC.strip()}" if NTFY_TOPIC.strip()
      else "Notifications    : disabled")

In [ ]:
# 2. Platform checks, helpers, notifications and secrets
import atexit
import codecs
import datetime
import json
import os
import shutil
import signal
import subprocess
import sys
import threading
import urllib.request
from pathlib import Path

CR, LF = chr(13), chr(10)

# --------------------------------------------------------------------------- #
# Platform checks (fail fast, before anything slow runs)
# --------------------------------------------------------------------------- #
ON_KAGGLE = Path("/kaggle").is_dir() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if not ON_KAGGLE:
    raise RuntimeError("This notebook targets Kaggle. On Google Colab use google-colab.ipynb.")
RUN_TYPE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown")  # Interactive | Batch (commit)

# Shared steps (doctor, previews) see only the first lane's GPU. Every lane step
# overrides CUDA_VISIBLE_DEVICES with its own GPU, so each training process sees
# exactly one device and HF Trainer never wraps the model in nn.DataParallel.
os.environ["CUDA_VISIBLE_DEVICES"] = LANE_SPECS[0]["gpu"]


def _query_gpus():
    """Returns {index: name} for physical GPUs (nvidia-smi ignores CUDA_VISIBLE_DEVICES)."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30,
        ).stdout
    except Exception:
        return {}
    gpus = {}
    for line in out.splitlines():
        if "," in line:
            idx, name = line.split(",", 1)
            gpus[idx.strip()] = name.strip()
    return gpus


GPUS = _query_gpus()
if not GPUS:
    raise RuntimeError("No GPU visible. Settings > Accelerator > GPU T4 x2 "
                       "(requires a phone-verified Kaggle account).")
for lane in LANE_SPECS:
    gpu_name = GPUS.get(lane["gpu"])
    if gpu_name is None:
        raise RuntimeError(f"Lane {lane['key']} wants GPU {lane['gpu']}, but only GPUs "
                           f"{sorted(GPUS)} exist. Two lanes need the GPU T4 x2 accelerator.")
    if "P100" in gpu_name:
        raise RuntimeError("Tesla P100 (Pascal, sm_60) selected. Recent PyTorch CUDA builds are "
                           "dropping Pascal support and the configs are tuned for a T4. "
                           "Switch the accelerator to GPU T4 x2.")
    if "T4" not in gpu_name:
        print(f"WARNING: lane {lane['key']} expected a Tesla T4 on GPU {lane['gpu']}, "
              f"found {gpu_name}. Configs are sized for 16 GB fp16.")
    lane["gpu_name"] = gpu_name


def _internet_ok(url="https://huggingface.co", timeout=15):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            return resp.status < 500
    except Exception:
        return False


if not _internet_ok():
    raise RuntimeError("No internet access. Settings > Internet > On "
                       "(requires a phone-verified Kaggle account).")


def _writable_dir(path):
    try:
        p = Path(path)
        p.mkdir(parents=True, exist_ok=True)
        probe = p / ".t2c-write-test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return True
    except Exception:
        return False


def _free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return float("nan")


WORK_ROOT = Path(SCRATCH_DIR) if _writable_dir(SCRATCH_DIR) else Path("/tmp")
REPO_DIR = WORK_ROOT / "capstone"
# Each lane runs from its own working directory, so per-process caches written
# to the cwd (e.g. Unsloth's compiled cache) never collide between lanes.
LANES_DIR = WORK_ROOT / "lanes"

# --------------------------------------------------------------------------- #
# Session, log locations and secrets
# --------------------------------------------------------------------------- #
SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S") + "-kaggle"
LOG_DIR = Path(OUTPUT_DIR) / "text2cypher-logs" / INSTANCE_LABEL / SESSION
LOG_DIR.mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as exc:
    # Scratch storage is wiped when the session ends, so training without Hub
    # sync would lose all progress. Refuse to start instead.
    raise RuntimeError(
        f"Could not read the 'HF_TOKEN' Kaggle secret ({type(exc).__name__}). "
        "Add-ons > Secrets: add HF_TOKEN (write access) and attach it to this notebook."
    ) from exc

os.environ["T2C_HUB_REPO"] = HUB_REPO
# Reduce PyTorch memory fragmentation on the 16 GB T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Child Python processes flush output immediately, so logs stay current.
os.environ["PYTHONUNBUFFERED"] = "1"

# --------------------------------------------------------------------------- #
# Pipeline state
# --------------------------------------------------------------------------- #
STATUS = {
    "platform": "kaggle",
    "run_type": RUN_TYPE,
    "instance": INSTANCE_LABEL,
    "session": SESSION,
    "gpus": GPUS,
    # None while running; then completed | failed | interrupted | runtime-ended.
    "outcome": None,
    "shared": {"current_step": None, "step_started": None, "last_line": "", "steps": []},
    "lanes": {
        lane["key"]: {
            "model": lane["model"], "variant": lane["variant"], "gpu": lane["gpu"],
            "gpu_name": lane["gpu_name"], "config": lane["config"], "run_name": None,
            "current_step": None, "step_started": None, "last_line": "", "steps": [],
            "train_limit": None, "export": None,
            # None while running; then completed | paused | no-time | failed | interrupted.
            "outcome": None,
        }
        for lane in LANE_SPECS
    },
}
_status_lock = threading.RLock()
_hub_lock = threading.Lock()
_hub_state = {"repo_ready": False}
STOP_EVENT = threading.Event()
ACTIVE_PROCS = {}
_procs_lock = threading.Lock()


def _scope(lane_key):
    return STATUS["lanes"][lane_key] if lane_key else STATUS["shared"]


def save_status():
    """Writes pipeline_status.json locally (saved with the version; never uploaded)."""
    with _status_lock:
        try:
            text = json.dumps(STATUS, indent=2, default=str)
        except RuntimeError:
            return  # A lane thread mutated STATUS mid-dump; the next save catches up.
        (LOG_DIR / "pipeline_status.json").write_text(text, encoding="utf-8")


# --------------------------------------------------------------------------- #
# ntfy push notifications
# --------------------------------------------------------------------------- #
NTFY_URL = f"{NTFY_SERVER.rstrip('/')}/{NTFY_TOPIC.strip()}" if NTFY_TOPIC.strip() else None


def _title(label, text):
    return f"T2C Kaggle {label}: {text}"


def notify(title, message, priority="default", tags=(), click=None):
    """Best-effort push notification via ntfy. Never raises.

    The topic is public: anyone who knows its name can read it, so only
    status text and log tails are sent, never tokens.
    """
    if not NTFY_URL:
        return False
    headers = {
        # HTTP header values must be latin-1; keep the title plain ASCII.
        "Title": title.encode("ascii", "replace").decode("ascii"),
        "Priority": str(priority),
    }
    if tags:
        headers["Tags"] = ",".join(tags)
    if click:
        headers["Click"] = click
    # ntfy turns bodies over 4096 bytes into attachments; keep the end of the text.
    body = message if len(message) <= 3000 else "..." + message[-3000:]
    req = urllib.request.Request(NTFY_URL, data=body.encode("utf-8"),
                                 headers=headers, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            resp.read()
        return True
    except Exception as exc:
        print(f"[ntfy] Notification failed: {exc}", file=sys.stderr)
        return False


def _log_tail(path, n=15):
    """Returns the last n lines of a step log (empty string if unreadable)."""
    try:
        lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    except Exception:
        return ""
    return LF.join(lines[-n:])


# --------------------------------------------------------------------------- #
# Hub uploads (results and GGUF only: one commit per call, never logs)
# --------------------------------------------------------------------------- #
def hub_upload(local_path, path_in_repo, message):
    """Uploads a file or folder to the Hub repo in a single commit. Never raises."""
    if not HUB_REPO or not os.environ.get("HF_TOKEN"):
        return False
    local_path = Path(local_path)
    if not local_path.exists():
        return False
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=os.environ["HF_TOKEN"])
        with _hub_lock:
            if not _hub_state["repo_ready"]:
                api.create_repo(HUB_REPO, repo_type="model", private=True, exist_ok=True)
                _hub_state["repo_ready"] = True
            if local_path.is_dir():
                api.upload_folder(folder_path=str(local_path), path_in_repo=path_in_repo,
                                  repo_id=HUB_REPO, repo_type="model", commit_message=message)
            else:
                api.upload_file(path_or_fileobj=str(local_path), path_in_repo=path_in_repo,
                                repo_id=HUB_REPO, repo_type="model", commit_message=message)
        return True
    except Exception as exc:
        print(f"[hub] Upload of {local_path.name} failed: {exc}", file=sys.stderr)
        return False


def archive_results(lane_key, paths):
    """Copies result files into the lane's results folder (saved output, no upload)."""
    dest = LOG_DIR / lane_key / "results"
    dest.mkdir(parents=True, exist_ok=True)
    for p in map(Path, paths):
        if not p.exists():
            continue
        if p.is_dir():
            shutil.copytree(p, dest / p.name, dirs_exist_ok=True)
        else:
            shutil.copy2(p, dest / p.name)


def publish_results(lane_key):
    """Uploads the lane's results folder plus its status to the Hub in ONE commit."""
    dest = LOG_DIR / lane_key / "results"
    dest.mkdir(parents=True, exist_ok=True)
    with _status_lock:
        lane_status = json.dumps(STATUS["lanes"][lane_key], indent=2, default=str)
    (dest / "lane_status.json").write_text(lane_status, encoding="utf-8")
    if not UPLOAD_RESULTS_TO_HUB:
        return False
    ok = hub_upload(dest, f"results/{lane_key}/{SESSION}", f"[{lane_key}] results {SESSION}")
    print(f"[{lane_key}] Results upload: {'OK' if ok else 'FAILED (kept in saved output)'}", flush=True)
    return ok


# --------------------------------------------------------------------------- #
# Step runner
# --------------------------------------------------------------------------- #
def _clean_lines(buf, text):
    """Splits streamed output into finished lines. Only the last carriage-return
    segment of each line is kept, so tqdm redraws do not bloat the log file."""
    text = buf + text.replace(CR + LF, LF)
    parts = text.split(LF)
    lines = [p.split(CR)[-1] for p in parts[:-1]]
    return lines, parts[-1].split(CR)[-1]


TRAIN_EXIT_HINTS = {
    2: "The config fingerprint differs from the remote checkpoint. Change run_name, or delete the old run on the Hub.",
    3: "Training loss became non-finite (fp16 overflow); no final adapter was saved.",
    4: "Training reached its time budget and checkpointed to the Hub. Commit a new version "
       "(Save Version > Save & Run All): training resumes automatically.",
    5: "Training finished but the final adapter did NOT reach the Hub (it exists only in this session). "
       "The lane retries once with `checkpoint push-final`; if that fails, the next session resumes from "
       "the end-of-training checkpoint and writes the final adapter again.",
}


def _interrupt_proc(proc):
    """Sends SIGINT to a step's process group (training writes an emergency checkpoint)."""
    try:
        os.killpg(proc.pid, signal.SIGINT)
        return proc.wait(timeout=180)
    except Exception:
        proc.kill()
        return proc.wait()


def _stop_all_procs():
    """Signals every running step; the owning threads collect the exit codes."""
    with _procs_lock:
        procs = list(ACTIVE_PROCS.values())
    for proc in procs:
        try:
            os.killpg(proc.pid, signal.SIGINT)
        except Exception:
            pass


def run_step(name, cmd, lane=None, allow_fail=False, env=None, cwd=None, echo=True):
    """Runs a shell command and writes its output to a log file.

    Shared steps (lane=None) also stream to the cell. Lane steps run in worker
    threads with echo=False; the heartbeat in cell 7 shows their progress.
    """
    if STOP_EVENT.is_set():
        raise KeyboardInterrupt(f"Stop requested before step '{name}'.")
    scope = _scope(lane)
    label = lane or "shared"
    with _status_lock:
        index = len(scope["steps"]) + 1
        scope["current_step"] = name
        scope["step_started"] = time.time()
        scope["last_line"] = ""
    log_dir = LOG_DIR / label
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{index:02d}-{name}.log"
    workdir = cwd or (str(REPO_DIR) if REPO_DIR.exists() else str(WORK_ROOT))
    started = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")

    banner = f"[{label}] [{index:02d}] {name}: {cmd}"
    if echo:
        print("=" * 72)
        print(banner)
        print(f"     log: {log_path}")
        print("=" * 72, flush=True)
    else:
        print(f"{banner}" + LF + f"     log: {log_path}", flush=True)

    t0 = time.time()
    buf = ""
    interrupted = False
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    proc_key = (label, index)
    with open(log_path, "w", encoding="utf-8") as fh:
        fh.write(f"$ {cmd}" + LF + f"# cwd: {workdir}" + LF + f"# started: {started}" + LF + LF)
        fh.flush()
        proc = subprocess.Popen(
            cmd, shell=True, cwd=workdir, env={**os.environ, **(env or {})},
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True,
        )
        with _procs_lock:
            ACTIVE_PROCS[proc_key] = proc
        try:
            while True:
                chunk = proc.stdout.read1(8192)
                if not chunk:
                    break
                text = decoder.decode(chunk)
                if echo:
                    sys.stdout.write(text)
                    sys.stdout.flush()
                lines, buf = _clean_lines(buf, text)
                for line in lines:
                    fh.write(line + LF)
                fh.flush()
                current = buf.strip() or (lines[-1].strip() if lines else "")
                if current:
                    scope["last_line"] = current
            rc = proc.wait()
        except KeyboardInterrupt:
            # Main-thread (shared) step: forward the interrupt to the child.
            interrupted = True
            rc = _interrupt_proc(proc)
        finally:
            with _procs_lock:
                ACTIVE_PROCS.pop(proc_key, None)
        if buf:
            fh.write(buf + LF)
        elapsed = time.time() - t0
        fh.write(LF + f"# exit code: {rc}" + LF + f"# elapsed: {elapsed:.1f}s" + LF)

    # Lane steps are stopped by the main thread via STOP_EVENT + SIGINT.
    interrupted = interrupted or STOP_EVENT.is_set()
    with _status_lock:
        scope["current_step"] = None
        scope["step_started"] = None
        scope["steps"].append({
            "index": index, "name": name, "cmd": cmd, "exit_code": rc,
            "elapsed_s": round(elapsed, 1), "started": started, "log": str(log_path),
            "interrupted": interrupted,
        })
    save_status()

    outcome = "OK" if rc == 0 else f"FAILED (exit {rc})"
    print(f"[{label}] [{index:02d}] {name}: {outcome} in {elapsed / 60:.1f} min", flush=True)

    if interrupted:
        if lane:
            scope["outcome"] = "interrupted"
        save_status()
        raise KeyboardInterrupt(f"Step '{name}' ({label}) interrupted.")

    if rc != 0 and not allow_fail:
        hint = TRAIN_EXIT_HINTS.get(rc) if name == "train" else None
        tail = _log_tail(log_path)
        paused = name == "train" and rc == 4
        if paused:
            scope["outcome"] = "paused"
            notify(_title(label, "paused at time limit"), LF.join([
                hint,
                f"Session {SESSION}, trained for {elapsed / 60:.0f} min.",
                "",
                tail,
            ]), priority="high", tags=["hourglass"])
        else:
            if lane:
                scope["outcome"] = "failed"
            notify(_title(label, f"step '{name}' failed"), LF.join([
                f"Exit code {rc} after {elapsed / 60:.1f} min. {hint or ''}".strip(),
                f"Session {SESSION}. Last output:",
                tail,
            ]), priority="urgent", tags=["rotating_light"])
        save_status()
        error = RuntimeError(f"Step '{name}' ({label}) failed with exit code {rc}. "
                             f"{hint or f'See {log_path}'}")
        # Already reported; stops other handlers from sending a duplicate alert.
        error.ntfy_sent = True
        # Lets a lane treat a time-limit pause as a clean halt.
        error.halt = "paused" if paused else None
        error.exit_code = rc
        raise error
    return rc


# --------------------------------------------------------------------------- #
# Crash / shutdown hooks
# --------------------------------------------------------------------------- #
def _notify_unfinished(reason):
    """Sends one alert if the kernel goes away before the pipeline reached an outcome."""
    if STATUS.get("outcome") is not None:
        return
    STATUS["outcome"] = "runtime-ended"
    running = [f"{key}: {st['current_step']}" for key, st in STATUS["lanes"].items()
               if st.get("current_step")]
    if STATUS["shared"].get("current_step"):
        running.insert(0, f"shared: {STATUS['shared']['current_step']}")
    where = ("Running: " + "; ".join(running)) if running else "No step was running."
    notify(_title(INSTANCE_LABEL, "runtime ended"), LF.join([
        f"{reason} before the pipeline finished.",
        where,
        f"Session {SESSION}. Hub checkpoints are kept; commit a new version to resume.",
    ]), priority="urgent", tags=["skull"])


def _on_sigterm(signum, frame):
    _notify_unfinished("The Kaggle session was shut down (SIGTERM)")
    # Give running trainers the chance to write an emergency checkpoint.
    STOP_EVENT.set()
    _stop_all_procs()
    prev = globals().get("_NTFY_PREV_SIGTERM")
    if callable(prev):
        prev(signum, frame)
    else:
        signal.signal(signal.SIGTERM, signal.SIG_DFL)
        os.kill(os.getpid(), signal.SIGTERM)


def _on_cell_exception(shell, etype, evalue, tb, tb_offset=None):
    """Alerts on any uncaught cell error that run_step has not already reported."""
    if not getattr(evalue, "ntfy_sent", False):
        STATUS["outcome"] = "failed"
        try:
            save_status()
        except Exception:
            pass
        notify(_title(INSTANCE_LABEL, "notebook error"),
               f"{etype.__name__}: {evalue}" + LF + f"Session {SESSION}.",
               priority="urgent", tags=["rotating_light"])
    elif STATUS.get("outcome") is None:
        STATUS["outcome"] = "failed"
        save_status()
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)


# Install the exit hooks once per kernel, even if this cell is re-run.
if not globals().get("_NTFY_HOOKS_INSTALLED"):
    _NTFY_PREV_SIGTERM = signal.getsignal(signal.SIGTERM)
    signal.signal(signal.SIGTERM, _on_sigterm)
    atexit.register(lambda: _notify_unfinished("The notebook kernel exited"))
    _NTFY_HOOKS_INSTALLED = True
get_ipython().set_custom_exc((Exception,), _on_cell_exception)


save_status()
min_free = 40 if len(LANE_SPECS) == 1 else 60
print(f"Platform   : Kaggle ({RUN_TYPE} session)")
print(f"GPUs       : {GPUS}")
for lane in LANE_SPECS:
    print(f"Lane       : {lane['key']} -> GPU {lane['gpu']} ({lane['gpu_name']})")
print(f"Work root  : {WORK_ROOT} ({_free_gb(WORK_ROOT):.0f} GB free, not saved)")
print(f"Output dir : {OUTPUT_DIR} ({_free_gb(OUTPUT_DIR):.0f} GB free, saved with the version)")
if _free_gb(WORK_ROOT) < min_free:
    print(f"WARNING: under {min_free} GB free in the work root. Base models, the merged model and "
          "the f16 GGUF intermediate may not fit; consider RUN_EXPORT = False.")
print(f"Session    : {SESSION}")
print(f"Local logs : {LOG_DIR}")
if NTFY_URL:
    print(f"Notify     : {NTFY_URL}")
    notify(_title(INSTANCE_LABEL, "pipeline started"),
           f"Session {SESSION} ({RUN_TYPE}). Lanes: "
           + ", ".join(f"{lane['key']} on GPU {lane['gpu']}" for lane in LANE_SPECS) + ".",
           priority="low", tags=["rocket"])

In [ ]:
# 3. Clone or update the repository and resolve per-lane paths
if (REPO_DIR / ".git").exists():
    run_step("git-pull", f"git -C {REPO_DIR} pull --ff-only", allow_fail=True, cwd=str(WORK_ROOT))
else:
    run_step("git-clone", f"git clone {GITHUB_REPO} {REPO_DIR}", cwd=str(WORK_ROOT))
os.chdir(REPO_DIR)

import yaml

# Lanes run from their own cwd, so every path passed to app.py is absolute.
APP = f"python {REPO_DIR / 'app.py'}"
ARTIFACTS = REPO_DIR / "artifacts"

for lane in LANE_SPECS:
    config_path = REPO_DIR / lane["config"]
    run_name = yaml.safe_load(config_path.read_text(encoding="utf-8"))["run_name"]
    lane["config_path"] = str(config_path)
    lane["run_name"] = run_name
    lane["run_dir"] = ARTIFACTS / "runs" / run_name
    lane["adapter_dir"] = ARTIFACTS / "runs" / run_name / "final"
    lane["pred_path"] = ARTIFACTS / "predictions" / f"{run_name}-test.jsonl"
    lane["report_path"] = ARTIFACTS / "reports" / f"{run_name}-eval.md"
    lane["merged_dir"] = ARTIFACTS / "merged" / run_name
    lane["workdir"] = LANES_DIR / lane["key"]
    lane["workdir"].mkdir(parents=True, exist_ok=True)
    STATUS["lanes"][lane["key"]]["run_name"] = run_name

if len({lane["run_name"] for lane in LANE_SPECS}) != len(LANE_SPECS):
    raise RuntimeError("Two lanes resolve to the same run_name; their artifacts would collide.")
save_status()

for lane in LANE_SPECS:
    print(f"[{lane['key']}] run_name={lane['run_name']}")
    print(f"    adapter : {lane['adapter_dir']}")
    print(f"    report  : {lane['report_path']}")
    print(f"    cwd     : {lane['workdir']}")

In [ ]:
# 4. Install dependencies (colab_bootstrap.sh is platform-neutral: pip + hf CLI check)
run_step("bootstrap", "bash scripts/colab_bootstrap.sh",
         env={"INSTALL_UNSLOTH": "1" if INSTALL_UNSLOTH else "0"})

In [ ]:
# 5. Download the dataset and the base model(s), one at a time
for target in DOWNLOAD_ORDER:
    run_step(f"download-{target}", f"bash download_data.sh {target}")

In [ ]:
# 6. Diagnostics and data previews (shared, sequential)
run_step("doctor", f"{APP} doctor", allow_fail=True)
for lane in LANE_SPECS:
    run_step(f"data-preview-{lane['key']}",
             f"{APP} data preview --config {lane['config_path']} --index 5", allow_fail=True)
    run_step(f"token-stats-{lane['key']}",
             f"{APP} data token-stats --config {lane['config_path']} --samples 500", allow_fail=True)

In [ ]:
# 7. Run the lanes in parallel: train -> predict -> evaluate -> export, one GPU each
import traceback

# Merge + GGUF run one lane at a time: merged fp16 weights and f16 GGUF
# intermediates are large, and two lanes building llama.cpp in the same
# checkout would collide.
_export_lock = threading.Lock()
LLAMA_QUANTIZE = REPO_DIR / "vendor" / "llama.cpp" / "build" / "bin" / "llama-quantize"


def _lane_step(lane, name, cmd, allow_fail=False, env=None):
    lane_env = {"CUDA_VISIBLE_DEVICES": lane["gpu"], **(env or {})}
    return run_step(name, cmd, lane=lane["key"], allow_fail=allow_fail,
                    env=lane_env, cwd=str(lane["workdir"]), echo=False)


def _report_lines(lane):
    report_json = Path(lane["report_path"]).with_suffix(".json")
    if not report_json.exists():
        return []
    try:
        overall = json.loads(report_json.read_text(encoding="utf-8"))[0].get("overall", {})
    except Exception as exc:
        return [f"(could not read evaluation report: {exc})"]
    out = []
    for key, label in (("n", "N"), ("google_bleu_corpus", "GLEU"),
                       ("exact_match", "Exact match"),
                       ("normalised_exact_match", "Norm. EM"),
                       ("execution_accuracy", "Exec. acc.")):
        if key in overall:
            out.append(f"{label}: {overall[key]}")
    return out


def _train(lane, st):
    """Returns True when a final adapter is available, False when the lane halted."""
    key = lane["key"]
    marker = lane["adapter_dir"] / "adapter_config.json"
    if SKIP_TRAINING_IF_FINAL_EXISTS and not marker.exists():
        # Exit 1 = no final adapter on the Hub yet. Exit 6 = Hub error: refuse to
        # train, otherwise a finished run would be retrained (history/030.md).
        rc = _lane_step(lane, "pull-final",
                        f"{APP} checkpoint pull --config {lane['config_path']} --step final",
                        allow_fail=True)
        if rc not in (0, 1):
            raise RuntimeError(f"[{key}] Could not check the Hub for a final adapter (exit {rc}). "
                               "Refusing to train; see the pull-final log.")
    if SKIP_TRAINING_IF_FINAL_EXISTS and marker.exists():
        print(f"[{key}] Final adapter found; skipping training.", flush=True)
        return True

    elapsed_min = (time.time() - PIPELINE_T0) / 60
    available_min = SESSION_BUDGET_MINUTES - elapsed_min - RESERVE_MINUTES
    train_limit = int(min(TIME_LIMIT_MINUTES, available_min))
    st["train_limit"] = train_limit
    print(f"[{key}] Session time used: {elapsed_min:.0f} min. Training limit: {train_limit} min.",
          flush=True)

    if train_limit < 15:
        st["outcome"] = "no-time"
        save_status()
        notify(_title(key, "not enough session time"),
               f"Only {available_min:.0f} min left for training in session {SESSION}. "
               "Commit a new version to continue.", priority="high", tags=["hourglass"])
        return False

    stop_at = (datetime.datetime.now(datetime.timezone.utc)
               + datetime.timedelta(minutes=train_limit))
    notify(_title(key, "training started"),
           f"Run {lane['run_name']} on GPU {lane['gpu']}, session {SESSION}, limit {train_limit} min. "
           f"The training clock starts after model loading and tokenization, so if the schedule "
           f"does not finish first, expect a 'paused at time limit' alert roughly 10-30 min after "
           f"{stop_at:%H:%M} UTC. Silence well past that means the session was killed.",
           priority="low", tags=["hourglass_flowing_sand"])
    try:
        _lane_step(lane, "train", f"{APP} train --config {lane['config_path']} --resume auto "
                                  f"--time-limit {train_limit}",
                   env={"T2C_SESSION": SESSION})  # Hub telemetry folder = this session
    except RuntimeError as exc:
        if getattr(exc, "halt", None) == "paused":
            print(f"[{key}] {exc}", flush=True)
            return False
        if getattr(exc, "exit_code", None) == 5:
            rc = _lane_step(lane, "push-final",
                            f"{APP} checkpoint push-final --config {lane['config_path']}",
                            allow_fail=True)
            if rc == 0:
                notify(_title(key, "final adapter uploaded on retry"),
                       f"push-final succeeded after the train step exited 5. Session {SESSION}.",
                       tags=["white_check_mark"])
                return True
        raise
    finally:
        archive_results(key, [
            lane["run_dir"] / "metrics.jsonl",
            lane["run_dir"] / "run_config.json",
            lane["run_dir"] / "final" / "run_metadata.json",
        ])
    return True


def _predict_and_evaluate(lane):
    key = lane["key"]
    pred = lane["pred_path"]
    report = lane["report_path"]
    _lane_step(lane, "predict",
               f"{APP} predict --config {lane['config_path']} --adapter {lane['adapter_dir']} "
               f"--split test --limit {PREDICT_LIMIT} --out {pred} --tag {lane['run_name']}")
    archive_results(key, [pred, pred.with_suffix(".meta.json")])

    eval_flags = "--execute --validate-syntax" if RUN_EXECUTION_EVAL else ""
    _lane_step(lane, "evaluate",
               f"{APP} evaluate --predictions {pred} {eval_flags} --out {report}")
    archive_results(key, [
        pred.with_name(pred.stem + ".scored.jsonl"),
        report,
        report.with_suffix(".json"),
    ])


def _export(lane, st):
    """Best-effort merge + GGUF conversion. Never raises for tool failures."""
    key = lane["key"]
    if not RUN_EXPORT:
        st["export"] = "disabled"
        return
    print(f"[{key}] Waiting for the export slot...", flush=True)
    with _export_lock:
        if STOP_EVENT.is_set():
            return
        merged = lane["merged_dir"]
        ok = _lane_step(lane, "merge", f"{APP} export merge --config {lane['config_path']} "
                                       f"--adapter {lane['adapter_dir']} --out {merged}",
                        allow_fail=True) == 0
        if ok and not LLAMA_QUANTIZE.exists():
            # A CPU build is enough for convert + quantize and compiles far faster than CUDA.
            ok = _lane_step(lane, "build-llama-cpp", f"bash {REPO_DIR / 'scripts' / 'build_llama_cpp.sh'}",
                            allow_fail=True, env={"BUILD_CUDA": "0"}) == 0
        if ok:
            ok = _lane_step(lane, "gguf", f"{APP} export gguf --merged {merged} --quant Q4_K_M",
                            allow_fail=True) == 0
        if ok:
            for gguf in sorted((ARTIFACTS / "gguf").glob(f"{lane['run_name']}-*.gguf*")):
                if gguf.suffix == ".json":
                    archive_results(key, [gguf])
                elif UPLOAD_GGUF:
                    # The GGUF stays in scratch storage (not saved output); the Hub is its only copy.
                    print(f"[{key}] Uploading {gguf.name} ({gguf.stat().st_size / 1e9:.2f} GB)...",
                          flush=True)
                    hub_upload(gguf, f"gguf/{lane['run_name']}/{gguf.name}", f"[{key}] {gguf.name}")
        st["export"] = "ok" if ok else "failed"
        if not KEEP_MERGED and merged.exists():
            shutil.rmtree(merged, ignore_errors=True)
            print(f"[{key}] Removed merged weights to free disk space.", flush=True)
        save_status()


def lane_pipeline(lane):
    key = lane["key"]
    st = STATUS["lanes"][key]
    try:
        delay = lane.get("stagger_s", 0)
        while delay > 0 and not STOP_EVENT.is_set():
            time.sleep(min(5, delay))
            delay -= 5
        if STOP_EVENT.is_set():
            raise KeyboardInterrupt("Stop requested before the lane started.")

        if _train(lane, st):
            _predict_and_evaluate(lane)
            _export(lane, st)
            st["outcome"] = "completed"
            save_status()
            lines = [f"Run {lane['run_name']} finished in session {SESSION}."] + _report_lines(lane)
            if st.get("export") == "failed":
                lines.append("GGUF export failed (best-effort); see the lane logs.")
            notify(_title(key, "lane finished"), LF.join(lines),
                   tags=["warning"] if st.get("export") == "failed" else ["white_check_mark"],
                   click=(f"https://huggingface.co/{HUB_REPO}/tree/main/results/{key}/{SESSION}"
                          if UPLOAD_RESULTS_TO_HUB and HUB_REPO else None))
    except KeyboardInterrupt:
        st["outcome"] = st.get("outcome") or "interrupted"
    except Exception as exc:
        st["outcome"] = "failed"
        tb_text = traceback.format_exc()
        print(f"[{key}] Lane failed:" + LF + tb_text, flush=True)
        if not getattr(exc, "ntfy_sent", False):
            notify(_title(key, "lane error"),
                   f"{type(exc).__name__}: {exc}" + LF + f"Session {SESSION}." + LF + tb_text[-1500:],
                   priority="urgent", tags=["rotating_light"])
    finally:
        save_status()
        publish_results(key)


def _gpu_usage():
    """Returns {index: 'used/total GiB, util%'} from nvidia-smi."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,memory.used,memory.total,utilization.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=20,
        ).stdout
    except Exception:
        return {}
    usage = {}
    for line in out.splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) == 4:
            idx, used, total, util = parts
            try:
                usage[idx] = f"{float(used) / 1024:.1f}/{float(total) / 1024:.1f} GiB {util}%"
            except ValueError:
                pass
    return usage


def _ram_available_gb():
    try:
        for line in Path("/proc/meminfo").read_text().splitlines():
            if line.startswith("MemAvailable:"):
                return int(line.split()[1]) / 1024 ** 2
    except Exception:
        pass
    return float("nan")


def heartbeat():
    usage = _gpu_usage()
    now = datetime.datetime.now(datetime.timezone.utc)
    elapsed = (time.time() - PIPELINE_T0) / 60
    lines = [f"--- {now:%H:%M:%S} UTC | session {elapsed:.0f}/{SESSION_BUDGET_MINUTES} min | "
             f"RAM available {_ram_available_gb():.1f} GB | disk free {_free_gb(WORK_ROOT):.0f} GB"]
    for lane in LANE_SPECS:
        st = STATUS["lanes"][lane["key"]]
        step = st.get("current_step")
        if step:
            started = st.get("step_started") or time.time()
            state = f"{step} {(time.time() - started) / 60:.0f} min"
        else:
            state = st.get("outcome") or "waiting"
        last = (st.get("last_line") or "")[-110:]
        lines.append(f"  [{lane['key']}] GPU{lane['gpu']} {usage.get(lane['gpu'], '?')} | {state} | {last}")
    print(LF.join(lines), flush=True)


threads = []
for i, lane in enumerate(LANE_SPECS):
    lane["stagger_s"] = i * LANE_STAGGER_S
    thread = threading.Thread(target=lane_pipeline, args=(lane,), name=f"lane-{lane['key']}", daemon=True)
    thread.start()
    threads.append(thread)

try:
    last_beat = time.time()
    while any(t.is_alive() for t in threads):
        time.sleep(5)
        if time.time() - last_beat >= HEARTBEAT_S:
            heartbeat()
            last_beat = time.time()
except KeyboardInterrupt:
    print("Interrupt received: stopping all lanes (training writes emergency checkpoints)...",
          flush=True)
    STOP_EVENT.set()
    _stop_all_procs()
    for t in threads:
        t.join(timeout=300)
    STATUS["outcome"] = "interrupted"
    save_status()
    notify(_title(INSTANCE_LABEL, "interrupted"),
           f"Session {SESSION} was stopped by the user.", priority="high", tags=["stop_sign"])
    raise

heartbeat()

In [ ]:
# 8. Summary
from IPython.display import Markdown, display

print(f"Session {SESSION}: instance {INSTANCE_LABEL} ({RUN_TYPE})")
print(f"Total session time: {(time.time() - PIPELINE_T0) / 60:.0f} min")
print(f"Local logs: {LOG_DIR}")

summary_lines = [f"Session {SESSION} finished."]
any_failed = False
for lane in LANE_SPECS:
    st = STATUS["lanes"][lane["key"]]
    outcome = st.get("outcome") or "unknown"
    any_failed = any_failed or outcome in ("failed", "unknown")
    print()
    print(f"[{lane['key']}] run_name={lane['run_name']} GPU {lane['gpu']} outcome={outcome} "
          f"export={st.get('export') or '-'}")
    for step in st["steps"]:
        print(f"  [{step['index']:02d}] {step['name']:<16} exit={step['exit_code']:<4} "
              f"{step['elapsed_s'] / 60:7.1f} min")
    metrics = _report_lines(lane)
    for line in metrics:
        print(f"  {line}")
    summary_lines.append(f"{lane['key']}: {outcome}" + (f" ({'; '.join(metrics)})" if metrics else ""))
    if UPLOAD_RESULTS_TO_HUB and HUB_REPO:
        print(f"  Results: https://huggingface.co/{HUB_REPO}/tree/main/results/{lane['key']}/{SESSION}")

STATUS["outcome"] = "completed"
save_status()
# Per-lane failures and pauses were already alerted; this is the end-of-session digest.
notify(_title(INSTANCE_LABEL, "pipeline finished"), LF.join(summary_lines),
       priority="high" if any_failed else "default",
       tags=["warning"] if any_failed else ["white_check_mark"])

for lane in LANE_SPECS:
    if Path(lane["report_path"]).exists():
        display(Markdown(Path(lane["report_path"]).read_text(encoding="utf-8")))